# 🧠 Transformer Block & Wrapping Up
**Course:** Deep Learning  
**Lecture:** 06 — Attention Mechanisms and Transformers  
**Part:** c — Transformer Block & Exam Wrap-Up

---

> **Where we are:** We have attention. Now we embed position information and wrap it in the residual Transformer block structure.

**What you'll learn:**
- Positional Encodings
- Layer Normalization
- Building the full Transformer Encoder Block
- Full course exam cheat sheet

**Exam relevance:** ⭐⭐⭐⭐⭐ (Putting it all together)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns
from tensorflow.keras import layers, Model

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Transformer Components

**Positional Encoding**
Transformers have no recurrent structure. They treat sequences as sets (bag of words). We must inject positional information.
$PE_{(pos,2i)} = \sin(pos / 10000^{2i/d_{model}})$
$PE_{(pos,2i+1)} = \cos(pos / 10000^{2i/d_{model}})$

**Layer Normalization vs Batch Normalization**
Transformers use LayerNorm.
- BatchNorm normalizes across the batch dimension. Fails for variable length sequences.
- LayerNorm normalizes across the feature dimension for each token independently.

**Transformer Encoder Block**
```
Input
  ↓
LayerNorm → Multi-Head Attention → Add
  ↓
LayerNorm → Feed Forward (Dense → ReLU → Dense) → Add
```
*(Modern Pre-LN architecture)*


## 🔀 Your Options

| Option | Positional Encoding | Pros | Cons |
|---|---|---|---|
| Sinusoidal | Math formula | Zero parameters, extrapolates well | Fixed, might not be optimal |
| Learned | `Embedding(max_len)` | Learns optimal positions | Fixed max length |
| RoPE | Rotary embeddings | Excellent long-context | Complex implementation |

At this step, you can choose between: Sinusoidal, Learned, or RoPE. We will explore Sinusoidal and Learned.


In [ ]:
# ✏️ Your Turn: Sinusoidal Positional Encoding

def get_positional_encoding(seq_len, d_model):
    # ### YOUR CODE HERE ###
    # Generate positional encoding matrix of shape (seq_len, d_model)
    # Use sin for even indices, cos for odd indices
    pass


In [ ]:
# ✅ Solution: Positional Encoding

def get_positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    
    # 2i / d_model -> floor(dims / 2) * 2 / d_model
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates
    
    # Apply sin to even indices, cos to odd indices
    pos_encoding = np.zeros(angle_rads.shape)
    pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])
    
    return tf.cast(pos_encoding, dtype=tf.float32)

pe = get_positional_encoding(50, 128)
plt.figure(figsize=(12, 6))
sns.heatmap(pe.numpy(), cmap='viridis')
plt.title("Sinusoidal Positional Encoding (Seq Len=50, d_model=128)")
plt.xlabel("Depth (d_model)")
plt.ylabel("Position (seq_len)")
plt.show()


In [ ]:
# ✏️ Your Turn & Solution: Transformer Block

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation='relu'),
            layers.Dense(d_model)
        ])
        
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def call(self, x, training):
        # Pre-LN architecture
        norm1 = self.layernorm1(x)
        attn_output = self.mha(norm1, norm1, norm1)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = x + attn_output
        
        norm2 = self.layernorm2(out1)
        ffn_output = self.ffn(norm2)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = out1 + ffn_output
        
        return out2

# Build dummy model
block = TransformerBlock(d_model=64, num_heads=4, dff=256)
x = tf.random.normal((1, 20, 64))
print("Block output shape:", block(x, training=False).shape)


## 🎓 Full Course Exam Cheat Sheet

- **Perceptron Update Rule:** $w \leftarrow w + \eta (y - \hat{y}) x$
- **Backprop Chain Rule:** $\frac{\partial L}{\partial w} = \frac{\partial L}{\partial y} \frac{\partial y}{\partial z} \frac{\partial z}{\partial w}$
- **CNN Output Size:** $O = \lfloor \frac{W - K + 2P}{S} \rfloor + 1$
- **LSTM Gates:** Input, Forget, Output gates control flow to the Cell State $C_t$.
- **Attention:** $\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$
- **Common Exam Traps:** 
  - Using Softmax instead of Sigmoid for Multi-label classification.
  - Forgetting `return_sequences=True` when stacking RNNs.
  - Mixing up LayerNorm and BatchNorm in Transformers.


## 🎓 Exam Corner

### Question 1 (Conceptual)
What is the role of the Feed-Forward Network in a Transformer block? Why can't we just use the raw embeddings without positional encoding?

<details><summary>💡 Answer</summary>

- **FFN Role:** Attention mixes information across positions (tokens). The FFN processes the information at each position individually. It adds non-linearity and capacity (usually expanding the dimension by 4x before projecting back).
- **Positional Encoding:** Self-attention operations are permutation invariant. Without positional encoding, the model would treat "The dog bit the man" exactly the same as "The man bit the dog".

</details>
